In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -U langchain langchain-community langchain-huggingface langchain-core langchain-text-splitters chromadb sentence-transformers

In [3]:
import json
import os
import shutil
import time
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 1. 경로 설정
BASE_DIR = "/content/drive/MyDrive/DILAB/YB/DILAB/LCK_RAG_Experiment/data"
INPUT_FILE_PATH = os.path.join(BASE_DIR, "lck_final_complete.json")
DB_PATH = "/content/drive/MyDrive/DILAB/YB/DILAB/LCK_RAG_Experiment/VectorDB"

# 2. 데이터 로드
print(f"[데이터 로딩 시작]: {INPUT_FILE_PATH}")

if not os.path.exists(INPUT_FILE_PATH):
    print(f"경로 오류: {INPUT_FILE_PATH}")
else:
    with open(INPUT_FILE_PATH, "r", encoding="utf-8") as f:
        raw_data = json.load(f)

    documents = []
    for item in raw_data:
        text_content = f"제목: {item['title']}\n내용: {item['content']}"
        metadata = {
            "source": item["url"],
            "date": item["date"],
            "title": item["title"]
        }
        doc = Document(page_content=text_content, metadata=metadata)
        documents.append(doc)

    print(f"기사 로드 완료: 총 {len(documents)}개")

    # 3. 텍스트 청킹 (Chunking)
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"[청킹 완료]: 총 {len(split_docs)}개의 청크 생성")

    # 4. 임베딩 및 DB 저장
    print("[임베딩 모델 로딩 중...]")

    embedding_model = HuggingFaceEmbeddings(
        model_name="jhgan/ko-sroberta-multitask",
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True}
    )

    print(f"[백터 DB 저장 시작]: {DB_PATH}")

    # 기존 폴더 삭제 후 재생성
    if os.path.exists(DB_PATH):
        try:
            shutil.rmtree(DB_PATH)
            time.sleep(2)
        except OSError as e:
            print(e)

    os.makedirs(DB_PATH, exist_ok=True)

    try:
        # DB 생성 및 저장
        vectordb = Chroma.from_documents(
            documents=split_docs,
            embedding=embedding_model,
            persist_directory=DB_PATH
        )

        try:
            vectordb.persist()
        except:
            pass

        # 5. 데이터 검증
        print("\n실제로 데이터가 잘 들어갔는지 확인합니다...")

        # DB를 다시 불러와서 개수 세보기
        check_db = Chroma(persist_directory=DB_PATH, embedding_function=embedding_model)
        count = check_db._collection.count()

        print(f"최종 저장된 문서 개수: {count}개")

        if count > 0:
            print("검증 성공!")
        else:
            print("데이터가 0개입니다.")

    except Exception as e:
        print(e)

[데이터 로딩 시작]: /content/drive/MyDrive/DILAB/YB/DILAB/LCK_RAG_Experiment/data/lck_final_complete.json
기사 로드 완료: 총 247개
[청킹 완료]: 총 490개의 청크 생성
[임베딩 모델 로딩 중...]
[백터 DB 저장 시작]: /content/drive/MyDrive/DILAB/YB/DILAB/LCK_RAG_Experiment/VectorDB

실제로 데이터가 잘 들어갔는지 확인합니다...
최종 저장된 문서 개수: 490개
검증 성공!


/tmp/ipython-input-2966915677.py:77: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectordb.persist()
/tmp/ipython-input-2966915677.py:85: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  check_db = Chroma(persist_directory=DB_PATH, embedding_function=embedding_model)
